# AnnNet benchmark

Wall-time and memory profile across the user-facing API surface.

**Sections**

1. Setup & harness
2. Construction & mutation
3. AnnNet-level lookups & iteration
4. Attributes (single + bulk + slice + graph-level)
5. Views & subviews
6. Slices (full API)
7. Subgraphs & ops (matrix surface)
8. Composite key & stoichiometry
9. Multilayer math
10. Multilayer attrs & set operations
11. Layer subgraph & slice constructors
12. IO formats (binary + delimited + scverse + structural)
13. Backend bridges (build + algorithm dispatch)
14. Traversal
15. History (full API)
16. Results table & plots
17. Scaling probe

Empty outputs by default — run all to populate.


## 1. Setup & harness


In [ ]:
# ── Parameters (edit these) ───────────────────────────────────────────
N = 2_000  # vertex count for the main fixture
M = 8_000  # edge count for the main fixture
SEED = 0

N_HYPER = 200  # number of undirected hyperedges
N_ML = 150  # vertices per layer in multilayer fixture
M_ML_PER_LAYER = 400  # intra edges per layer
N_LAYERS = 2

SCALING_SIZES = [(500, 1_500), (2_000, 8_000), (5_000, 20_000)]

BENCH_IO = True
BENCH_SCVERSE = True  # needs anndata / mudata installed
BENCH_OPTIONAL_IO = True  # graphml/gexf/excel/csv

In [ ]:
import gc
import os
import time
import tempfile
import warnings
from pathlib import Path

import numpy as np

import annnet
from annnet import AnnNet

try:
    import psutil

    _proc = psutil.Process(os.getpid())

    def rss_mb() -> float:
        return _proc.memory_info().rss / (1024 * 1024)
except ImportError:

    def rss_mb() -> float:
        return float('nan')


print('annnet at:', annnet.__file__)
print('starting RSS (MB):', round(rss_mb(), 2))

In [ ]:
# bench harness — records wall + RSS delta into _results
_results: dict = {}


def bench(label, fn):
    gc.collect()
    rss0 = rss_mb()
    t0 = time.perf_counter()
    try:
        out = fn()
        elapsed = time.perf_counter() - t0
        rss1 = rss_mb()
        _results[label] = {'wall_s': elapsed, 'delta_rss_mb': rss1 - rss0, 'ok': True}
        print(f'{label:<55} {elapsed * 1000:>9.2f} ms   {rss1 - rss0:+7.1f} MB')
        return out
    except Exception as e:
        elapsed = time.perf_counter() - t0
        _results[label] = {'wall_s': elapsed, 'delta_rss_mb': 0.0, 'ok': False, 'err': str(e)}
        print(f'{label:<55} SKIPPED ({type(e).__name__}: {str(e)[:60]})')
        return None

## 2. Construction & mutation


In [ ]:
rng = np.random.default_rng(SEED)

v_ids = [f'v{i:06d}' for i in range(N)]
src_idx = rng.integers(0, N, size=M)
tgt_idx = rng.integers(0, N, size=M)
edge_rows = [
    {'source': v_ids[s], 'target': v_ids[t], 'edge_id': f'e{i:07d}', 'weight': float(rng.random())}
    for i, (s, t) in enumerate(zip(src_idx, tgt_idx, strict=False))
]
print(f'prepared {N:,} vertices / {M:,} edges')

In [ ]:
def _build_bulk():
    G = AnnNet(directed=True)
    G.add_vertices(v_ids)
    G.add_edges(edge_rows)
    return G


G = bench('construct (add_vertices + add_edges)', _build_bulk)
print('   nv =', G.nv, 'ne =', G.ne)

In [ ]:
SMALL = min(2000, M // 4)


def _build_singular():
    Gs = AnnNet(directed=True)
    Gs.add_vertices(v_ids[:SMALL])
    for r in edge_rows[:SMALL]:
        Gs.add_edges(r['source'], r['target'], edge_id=r['edge_id'], weight=r['weight'])
    return Gs


_ = bench(f'add_edges singular x{SMALL}', _build_singular)

In [ ]:
# Hyperedges
hyper_members = [
    [v_ids[i] for i in rng.choice(N, size=int(rng.integers(3, 7)), replace=False)]
    for _ in range(N_HYPER)
]
hyper_rows = [{'members': m, 'edge_id': f'h{i:05d}'} for i, m in enumerate(hyper_members)]


def _add_hypers():
    Gh = AnnNet(directed=False)
    Gh.add_vertices(v_ids)
    Gh.add_hyperedges_bulk(hyper_rows)
    return Gh


Gh = bench(f'add_hyperedges_bulk x{N_HYPER}', _add_hypers)
print('   hyper nv =', Gh.nv, 'ne =', Gh.ne)

In [ ]:
# Bulk + singular removal
to_drop = [r['edge_id'] for r in edge_rows[: M // 4]]
G_for_remove = _build_bulk()
_ = bench(f'remove_edges (bulk x{len(to_drop)})', lambda: G_for_remove.remove_edges(to_drop))

G_for_vremove = _build_bulk()
v_drop = v_ids[: N // 10]
_ = bench(f'remove_vertices (bulk x{len(v_drop)})', lambda: G_for_vremove.remove_vertices(v_drop))

# Singular variants (smaller batch)
G_rs = _build_bulk()
_ = bench(
    'remove_edge singular x100', lambda: [G_rs.remove_edge(r['edge_id']) for r in edge_rows[:100]]
)

G_rsv = _build_bulk()
_ = bench('remove_vertex singular x50', lambda: [G_rsv.remove_vertex(v) for v in v_ids[:50]])

In [ ]:
G_undir = _build_bulk()
_ = bench('make_undirected', G_undir.make_undirected)
_ = bench('ops.copy (deep)', G.ops.copy)

## 3. Lookups & iteration


In [ ]:
# Iteration over all vertices / edges
_ = bench('vertices() (full iter)', lambda: list(G.vertices()))
_ = bench('edges() (full iter)', lambda: list(G.edges()))
try:
    _ = bench('edge_list()', lambda: list(G.edge_list()))
except Exception as e:
    print('edge_list skipped:', e)

In [ ]:
sample_vid = v_ids[0]
sample_eid = edge_rows[0]['edge_id']

_ = bench('has_vertex (1 lookup)', lambda: G.has_vertex(sample_vid))
_ = bench('has_edge (1 lookup)', lambda: G.has_edge(edge_id=sample_eid))
_ = bench('get_vertex (by row idx)', lambda: G.get_vertex(0))
_ = bench('get_edge (1 lookup)', lambda: G.get_edge(sample_eid))
_ = bench('incident_edges (1 vertex)', lambda: list(G.incident_edges(sample_vid)))

## 4. Attributes


In [ ]:
v_attrs_payload = {
    vid: {'tier': int(rng.integers(0, 4)), 'value': float(rng.random())} for vid in v_ids[: N // 2]
}
e_attrs_payload = {
    row['edge_id']: {'confidence': float(rng.random())} for row in edge_rows[: M // 2]
}

_ = bench(
    'attrs.set_vertex_attrs_bulk (N/2)', lambda: G.attrs.set_vertex_attrs_bulk(v_attrs_payload)
)
_ = bench('attrs.set_edge_attrs_bulk (M/2)', lambda: G.attrs.set_edge_attrs_bulk(e_attrs_payload))

In [ ]:
# Per-call setters
G_sp = _build_bulk()
ATT_SMALL = min(500, N // 4)

_ = bench(
    f'attrs.set_vertex_attrs single x{ATT_SMALL}',
    lambda: [G_sp.attrs.set_vertex_attrs(vid, tier=1) for vid in v_ids[:ATT_SMALL]],
)
_ = bench(
    f'attrs.set_edge_attrs single x{ATT_SMALL}',
    lambda: [G_sp.attrs.set_edge_attrs(r['edge_id'], conf=0.5) for r in edge_rows[:ATT_SMALL]],
)

In [ ]:
# Lookups & extractors
_ = bench('attrs.get_attr_vertex (1)', lambda: G.attrs.get_attr_vertex(sample_vid, 'tier'))
_ = bench('attrs.get_attr_edge (1)', lambda: G.attrs.get_attr_edge(sample_eid, 'confidence'))
_ = bench('attrs.get_vertex_attrs (1 full dict)', lambda: G.attrs.get_vertex_attrs(sample_vid))
_ = bench('attrs.get_edge_attrs (1 full dict)', lambda: G.attrs.get_edge_attrs(sample_eid))
_ = bench('attrs.get_attr_vertices (all)', G.attrs.get_attr_vertices)
_ = bench('attrs.get_attr_edges (all)', G.attrs.get_attr_edges)
_ = bench('attrs.get_attr_from_edges (column)', lambda: G.attrs.get_attr_from_edges('confidence'))
_ = bench('attrs.get_edges_by_attr (filter)', lambda: G.attrs.get_edges_by_attr('confidence', 0.5))

In [ ]:
# Slice attrs (single + bulk)
G.slices.add('s1')
sample_eids = [r['edge_id'] for r in edge_rows[: M // 8]]
G.slices.add_edges('s1', sample_eids)

_ = bench(
    'attrs.set_edge_slice_attrs single x200',
    lambda: [G.attrs.set_edge_slice_attrs('s1', eid, weight=1.5) for eid in sample_eids[:200]],
)

slice_attrs = {eid: {'weight': float(rng.random())} for eid in sample_eids[:500]}
_ = bench(
    'attrs.set_edge_slice_attrs_bulk (500)',
    lambda: G.attrs.set_edge_slice_attrs_bulk('s1', slice_attrs),
)

_ = bench(
    'attrs.set_slice_edge_weight (1)',
    lambda: G.attrs.set_slice_edge_weight('s1', sample_eids[0], 2.0),
)
_ = bench(
    'attrs.get_effective_edge_weight (1)',
    lambda: G.attrs.get_effective_edge_weight(sample_eids[0], slice='s1'),
)

In [ ]:
# Graph-level (uns) attrs
_ = bench('attrs.set_graph_attribute', lambda: G.attrs.set_graph_attribute('study', 'demo'))
_ = bench('attrs.get_graph_attribute', lambda: G.attrs.get_graph_attribute('study'))
_ = bench('attrs.get_graph_attributes (all)', G.attrs.get_graph_attributes)
_ = bench('attrs.audit_attributes', G.attrs.audit_attributes)

## 5. Views & subviews


In [ ]:
_ = bench('views.edges()', G.views.edges)
_ = bench('views.vertices()', G.views.vertices)
_ = bench('views.slices()', G.views.slices)
_ = bench('views.aspects()', G.views.aspects)
_ = bench('views.layers()', G.views.layers)

In [ ]:
from annnet.core._Views import GraphView

sub_vertices = set(v_ids[: N // 10])
v_view = GraphView(G, vertices=sub_vertices)
_ = bench('GraphView.materialize (N/10 verts)', v_view.materialize)
_ = bench('GraphView.subview (new filter)', lambda: v_view.subview(vertices={v_ids[0], v_ids[1]}))

## 6. Slices (full API)


In [ ]:
G_sl = _build_bulk()
for i in range(5):
    G_sl.slices.add(f'sl{i}')
    G_sl.slices.add_edges(
        f'sl{i}', [r['edge_id'] for r in edge_rows[i * (M // 5) : (i + 1) * (M // 5)]]
    )

_ = bench('slices.list()', G_sl.slices.list)
_ = bench('slices.exists()', lambda: G_sl.slices.exists('sl0'))
_ = bench('slices.count()', G_sl.slices.count)
_ = bench('slices.active', lambda: G_sl.slices.active)
_ = bench('slices.edges("sl0")', lambda: G_sl.slices.edges('sl0'))
_ = bench('slices.vertices("sl0")', lambda: G_sl.slices.vertices('sl0'))
_ = bench('slices.union(sl0, sl1)', lambda: G_sl.slices.union(['sl0', 'sl1']))
_ = bench('slices.intersect(sl0, sl1)', lambda: G_sl.slices.intersect(['sl0', 'sl1']))
_ = bench('slices.difference(sl0, sl1)', lambda: G_sl.slices.difference('sl0', 'sl1'))

In [ ]:
# info / stats / aggregate / summary
_ = bench('slices.info("sl0")', lambda: G_sl.slices.info('sl0'))
_ = bench('slices.stats()', G_sl.slices.stats)
_ = bench('slices.summary()', G_sl.slices.summary)
_ = bench(
    'slices.aggregate (union x5 → sl_all)',
    lambda: G_sl.slices.aggregate(
        source_slice_ids=['sl0', 'sl1', 'sl2', 'sl3', 'sl4'],
        target_slice_id='sl_all',
        method='union',
    ),
)

In [ ]:
# Presence + specific
_ = bench('slices.vertex_presence (1 vertex)', lambda: G_sl.slices.vertex_presence(v_ids[0]))
_ = bench(
    'slices.edge_presence (1 edge)',
    lambda: G_sl.slices.edge_presence(edge_id=edge_rows[0]['edge_id']),
)
_ = bench('slices.specific_edges', lambda: G_sl.slices.specific_edges('sl0'))

In [ ]:
# Creation from operations
G_sc = _build_bulk()
for i in range(3):
    G_sc.slices.add(f'src{i}')
    G_sc.slices.add_edges(
        f'src{i}', [r['edge_id'] for r in edge_rows[i * (M // 3) : (i + 1) * (M // 3)]]
    )

_ = bench('slices.union_create', lambda: G_sc.slices.union_create(['src0', 'src1'], 'u_create'))
_ = bench(
    'slices.intersect_create', lambda: G_sc.slices.intersect_create(['src0', 'src1'], 'i_create')
)
_ = bench(
    'slices.difference_create', lambda: G_sc.slices.difference_create('src0', 'src1', 'd_create')
)

In [ ]:
# remove
G_rm = _build_bulk()
G_rm.slices.add('to_drop')
_ = bench('slices.remove()', lambda: G_rm.slices.remove('to_drop'))

## 7. Subgraphs & ops (matrix surface)


In [ ]:
sub_v = list(set(v_ids[: N // 4]))
sub_e = sample_eids[: M // 8]

_ = bench('ops.subgraph(vertices=N/4)', lambda: G.ops.subgraph(sub_v))
_ = bench(
    'ops.extract_subgraph(v=N/4, e=M/8)',
    lambda: G.ops.extract_subgraph(vertices=sub_v, edges=sub_e),
)
_ = bench('ops.edge_subgraph(e=M/8)', lambda: G.ops.edge_subgraph(sub_e))
_ = bench('ops.reverse', G.ops.reverse)
_ = bench('ops.memory_usage', G.ops.memory_usage)

In [ ]:
# Matrix accessors
_ = bench('ops.incidence', G.ops.incidence)
_ = bench('ops.vertex_incidence_matrix', G.ops.vertex_incidence_matrix)
_ = bench('ops.incidence_as_lists', G.ops.incidence_as_lists)
_ = bench('hash(G.ops)', lambda: hash(G.ops))

## 8. Composite key & stoichiometry


In [ ]:
# Composite key
G_ck = AnnNet(directed=True)
G_ck.add_vertices(v_ids[:200])
for vid in v_ids[:200]:
    G_ck.attrs.set_vertex_attrs(vid, symbol=vid.upper())
_ = bench('set_vertex_key(symbol)', lambda: G_ck.set_vertex_key('symbol'))
_ = bench(
    'get_or_create_vertex_by_attrs (new)', lambda: G_ck.get_or_create_vertex_by_attrs(symbol='NEW1')
)
_ = bench(
    'get_or_create_vertex_by_attrs (existing)',
    lambda: G_ck.get_or_create_vertex_by_attrs(symbol='V000000'),
)

In [ ]:
# Stoichiometry on a single hyperedge
G_st = AnnNet(directed=False)
G_st.add_vertices(['A', 'B', 'C', 'D'])
G_st.add_edges(src=['A', 'B'], tgt=['C', 'D'], edge_id='rxn', directed=True)
_ = bench(
    'set_edge_coeffs',
    lambda: G_st.set_edge_coeffs('rxn', {'A': -2.0, 'B': -1.0, 'C': 1.0, 'D': 1.0}),
)

## 9. Multilayer math


In [ ]:
def _build_ml():
    Gm = AnnNet(directed=False)
    layer_labels = [f'L{i}' for i in range(N_LAYERS)]
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        Gm.layers.set_aspects(['x'], {'x': layer_labels})
        sub_v = [f'm{i:05d}' for i in range(N_ML)]
        for L in layer_labels:
            Gm.add_vertices(sub_v, layer={'x': L})
        for L in layer_labels:
            for k in range(M_ML_PER_LAYER):
                s = sub_v[int(rng.integers(0, N_ML))]
                t = sub_v[int(rng.integers(0, N_ML))]
                Gm.add_edges((s, (L,)), (t, (L,)), edge_id=f'{L}_e{k:06d}', weight=1.0)
        for a, b in zip(layer_labels, layer_labels[1:], strict=False):
            Gm.layers.add_layer_coupling_pairs([((a,), (b,))])
    return Gm


Gm = bench(f'multilayer fixture build (N={N_ML} x L={N_LAYERS})', _build_ml)
print('  ml nv =', Gm.nv, '  ne =', Gm.ne)

In [ ]:
_ = bench('supra_adjacency', Gm.layers.supra_adjacency)
_ = bench('supra_degree', Gm.layers.supra_degree)
_ = bench('supra_laplacian (comb)', lambda: Gm.layers.supra_laplacian(kind='comb'))
_ = bench('supra_laplacian (norm)', lambda: Gm.layers.supra_laplacian(kind='norm'))
_ = bench('build_intra_block', Gm.layers.build_intra_block)
_ = bench('build_inter_block', Gm.layers.build_inter_block)
_ = bench('build_coupling_block', Gm.layers.build_coupling_block)
_ = bench('transition_matrix', Gm.layers.transition_matrix)
_ = bench(
    'supra_adjacency_scaled (ω=2)', lambda: Gm.layers.supra_adjacency_scaled(coupling_scale=2.0)
)
_ = bench('supra_incidence', Gm.layers.supra_incidence)

In [ ]:
import numpy as np

p0 = np.ones(Gm.nv) / Gm.nv
x0 = p0.copy()
_ = bench('random_walk_step', lambda: Gm.layers.random_walk_step(p0))
_ = bench('diffusion_step (tau=0.1)', lambda: Gm.layers.diffusion_step(x0, tau=0.1))

In [ ]:
_ = bench('algebraic_connectivity', Gm.layers.algebraic_connectivity)
_ = bench('k_smallest_laplacian_eigs (k=3)', lambda: Gm.layers.k_smallest_laplacian_eigs(k=3))
_ = bench('dominant_rw_eigenpair', Gm.layers.dominant_rw_eigenpair)
_ = bench(
    'sweep_coupling_regime (3 scales)', lambda: Gm.layers.sweep_coupling_regime([0.5, 1.0, 2.0])
)

In [ ]:
_ = bench('layer_degree_vectors', Gm.layers.layer_degree_vectors)
_ = bench('participation_coefficient', Gm.layers.participation_coefficient)
_ = bench('versatility', Gm.layers.versatility)

partition = np.zeros(Gm.nv, dtype=int)
_ = bench(
    'multislice_modularity (all-1 partition)', lambda: Gm.layers.multislice_modularity(partition)
)

In [ ]:
# Tensor view + flatten round-trip
view = bench('adjacency_tensor_view', Gm.layers.adjacency_tensor_view)
_ = bench('tensor_index', Gm.layers.tensor_index)
if view is not None:
    _ = bench('flatten_to_supra', lambda: Gm.layers.flatten_to_supra(view))
A_sup = Gm.layers.supra_adjacency()
_ = bench('unflatten_from_supra', lambda: Gm.layers.unflatten_from_supra(A_sup))

## 10. Multilayer attrs & set operations


In [ ]:
# Aspect / layer / vertex-layer / elementary-layer attrs (round-trip each)
_ = bench('set_aspect_attrs', lambda: Gm.layers.set_aspect_attrs('x', kind='temporal'))
_ = bench('get_aspect_attrs', lambda: Gm.layers.get_aspect_attrs('x'))

_ = bench('set_layer_attrs', lambda: Gm.layers.set_layer_attrs(('L0',), note='alpha'))
_ = bench('get_layer_attrs', lambda: Gm.layers.get_layer_attrs(('L0',)))

_ = bench(
    'set_vertex_layer_attrs',
    lambda: Gm.layers.set_vertex_layer_attrs('m00000', ('L0',), state='on'),
)
_ = bench('get_vertex_layer_attrs', lambda: Gm.layers.get_vertex_layer_attrs('m00000', ('L0',)))

_ = bench(
    'set_elementary_layer_attrs',
    lambda: Gm.layers.set_elementary_layer_attrs('x', 'L0', label='early'),
)
_ = bench('get_elementary_layer_attrs', lambda: Gm.layers.get_elementary_layer_attrs('x', 'L0'))

In [ ]:
# Layer set operations
_ = bench('layer_union', lambda: Gm.layers.layer_union([('L0',), ('L1',)]))
_ = bench('layer_intersection', lambda: Gm.layers.layer_intersection([('L0',), ('L1',)]))
_ = bench('layer_difference', lambda: Gm.layers.layer_difference(('L0',), ('L1',)))

In [ ]:
# Layer iteration helpers
_ = bench('iter_layers (consume)', lambda: list(Gm.layers.iter_layers()))
_ = bench('iter_vertex_layers (1 vertex)', lambda: list(Gm.layers.iter_vertex_layers('m00000')))
_ = bench('has_presence (1 query)', lambda: Gm.layers.has_presence('m00000', ('L0',)))
_ = bench(
    'layer_id_to_tuple ↔ layer_tuple_to_id round trip',
    lambda: Gm.layers.layer_tuple_to_id(Gm.layers.layer_id_to_tuple('L0')),
)

In [ ]:
# Additional coupling generators
G_cat = bench('multilayer fresh for couplings', _build_ml)
if G_cat is not None:
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        _ = bench(
            'add_diagonal_coupling_filter',
            lambda: G_cat.layers.add_diagonal_coupling_filter({'x': {'L0', 'L1'}}),
        )
        _ = bench(
            'add_categorical_coupling',
            lambda: G_cat.layers.add_categorical_coupling('x', [['L0', 'L1']]),
        )

## 11. Layer subgraph & slice constructors


In [ ]:
_ = bench('subgraph_from_layer_tuple', lambda: Gm.layers.subgraph_from_layer_tuple(('L0',)))
_ = bench(
    'subgraph_from_layer_union', lambda: Gm.layers.subgraph_from_layer_union([('L0',), ('L1',)])
)
_ = bench(
    'subgraph_from_layer_intersection',
    lambda: Gm.layers.subgraph_from_layer_intersection([('L0',), ('L1',)]),
)
_ = bench(
    'subgraph_from_layer_difference',
    lambda: Gm.layers.subgraph_from_layer_difference(('L0',), ('L1',)),
)

In [ ]:
G_csl = _build_ml()
if G_csl is not None:
    _ = bench(
        'create_slice_from_layer', lambda: G_csl.layers.create_slice_from_layer('csl_t', ('L0',))
    )
    _ = bench(
        'create_slice_from_layer_union',
        lambda: G_csl.layers.create_slice_from_layer_union('csl_u', [('L0',), ('L1',)]),
    )
    _ = bench(
        'create_slice_from_layer_intersection',
        lambda: G_csl.layers.create_slice_from_layer_intersection('csl_i', [('L0',), ('L1',)]),
    )
    _ = bench(
        'create_slice_from_layer_difference',
        lambda: G_csl.layers.create_slice_from_layer_difference('csl_d', ('L0',), ('L1',)),
    )
    _ = bench('flatten_layers', G_csl.layers.flatten_layers)

## 12. IO formats


In [ ]:
if BENCH_IO:
    tmp = Path(tempfile.mkdtemp(prefix='annnet_bench_'))
    print('temp dir:', tmp)

    from annnet import (
        to_parquet,
        from_parquet,
        to_json,
        from_json,
        write,
        read,
        to_dataframes,
        from_dataframes,
        to_cx2,
        from_cx2,
        to_sif,
        from_sif,
        write_ndjson,
    )

    _ = bench('IO: to_parquet', lambda: to_parquet(G, tmp / 'g_parquet'))
    _ = bench('IO: from_parquet', lambda: from_parquet(tmp / 'g_parquet'))
    _ = bench('IO: to_json', lambda: to_json(G, tmp / 'g.json'))
    _ = bench('IO: from_json', lambda: from_json(tmp / 'g.json'))
    _ = bench('IO: write (.annnet)', lambda: write(G, tmp / 'g.annnet'))
    _ = bench('IO: read (.annnet)', lambda: read(tmp / 'g.annnet'))
    _ = bench('IO: to_dataframes', lambda: to_dataframes(G))
    _ = bench('IO: write_ndjson', lambda: write_ndjson(G, tmp / 'g.ndjson'))

    dfs = to_dataframes(G)
    _ = bench('IO: from_dataframes (round-trip)', lambda: from_dataframes(dfs))

In [ ]:
if BENCH_IO:
    cx2 = bench('IO: to_cx2 (in-memory)', lambda: to_cx2(G))
    _ = bench('IO: from_cx2 (in-memory)', lambda: from_cx2(cx2))
    _ = bench('IO: to_sif', lambda: to_sif(G, tmp / 'g.sif'))
    _ = bench('IO: from_sif', lambda: from_sif(tmp / 'g.sif'))

In [ ]:
if BENCH_OPTIONAL_IO:
    from annnet import (
        edges_to_csv,
        hyperedges_to_csv,
        from_csv,
        to_graphml,
        from_graphml,
        to_gexf,
        from_gexf,
    )

    _ = bench('IO: edges_to_csv', lambda: edges_to_csv(G, tmp / 'g_edges.csv'))
    _ = bench('IO: from_csv (edge_list)', lambda: from_csv(tmp / 'g_edges.csv', schema='edge_list'))

    Gh_csv = _add_hypers()
    _ = bench('IO: hyperedges_to_csv', lambda: hyperedges_to_csv(Gh_csv, tmp / 'g_hyper.csv'))

    _ = bench('IO: to_graphml', lambda: to_graphml(G, tmp / 'g.graphml'))
    _ = bench('IO: from_graphml', lambda: from_graphml(tmp / 'g.graphml'))
    _ = bench('IO: to_gexf', lambda: to_gexf(G, tmp / 'g.gexf'))
    _ = bench('IO: from_gexf', lambda: from_gexf(tmp / 'g.gexf'))

In [ ]:
if BENCH_SCVERSE:
    try:
        from annnet import to_anndata, from_anndata

        adata = bench('IO: to_anndata', lambda: to_anndata(G))
        if adata is not None:
            _ = bench('IO: from_anndata', lambda: from_anndata(adata))
    except ImportError as e:
        print('anndata not installed, skipping:', e)

    try:
        from annnet import to_mudata, from_mudata

        mdata = bench('IO: to_mudata', lambda: to_mudata(G))
        if mdata is not None:
            _ = bench('IO: from_mudata', lambda: from_mudata(mdata))
    except ImportError as e:
        print('mudata not installed, skipping:', e)

    try:
        from annnet import to_spatialdata, from_spatialdata

        sd = bench('IO: to_spatialdata', lambda: to_spatialdata(G))
        if sd is not None:
            _ = bench('IO: from_spatialdata', lambda: from_spatialdata(sd))
    except ImportError as e:
        print('spatialdata not installed, skipping:', e)

## 13. Backend bridges


In [ ]:
_ = bench('nx.backend (build + cache)', G.nx.backend)
_ = bench('nx.backend (cached re-read)', G.nx.backend)
_ = bench('ig.backend (build + cache)', G.ig.backend)
_ = bench('ig.backend (cached re-read)', G.ig.backend)

In [ ]:
from annnet import to_nx, from_nx, to_igraph, from_igraph

G_rt = _build_bulk()
try:
    _ = bench('to_nx (build + manifest)', lambda: to_nx(G_rt))
    nxg, nx_manifest = to_nx(G_rt)
    _ = bench('from_nx (rebuild)', lambda: from_nx(nxg, nx_manifest))
except Exception as e:
    print('to_nx/from_nx skipped:', type(e).__name__, e)

try:
    _ = bench('to_igraph (build + manifest)', lambda: to_igraph(G_rt))
    igg, ig_manifest = to_igraph(G_rt)
    _ = bench('from_igraph (rebuild)', lambda: from_igraph(igg, ig_manifest))
except Exception as e:
    print('to_igraph/from_igraph skipped:', type(e).__name__, e)

In [ ]:
# Algorithm dispatch through the wrappers (the per-call overhead matters here)
import networkx as nx

nxG = G.nx.backend()
_ = bench(
    'G.nx.shortest_path_length(G, src) — direct nx algo',
    lambda: nx.shortest_path_length(nxG, source=v_ids[0]),
)
_ = bench(
    'G.nx (wrapped) shortest_path_length', lambda: G.nx.shortest_path_length(G, source=v_ids[0])
)
_ = bench('G.nx (wrapped) degree centrality', lambda: G.nx.degree_centrality(G))

## 14. Traversal


In [ ]:
sample_v = v_ids[0]
_ = bench('out_neighbors (1)', lambda: list(G.out_neighbors(sample_v)))
_ = bench('successors (1)', lambda: list(G.successors(sample_v)))
_ = bench('in_neighbors (1)', lambda: list(G.in_neighbors(sample_v)))
_ = bench('predecessors (1)', lambda: list(G.predecessors(sample_v)))
_ = bench('neighbors (1)', lambda: list(G.neighbors(sample_v)))

## 15. History (full API)


In [ ]:
G_h = _build_bulk()
_ = bench('history.enable(True)', lambda: G_h.history.enable(True))
G_h.add_vertices(['extra1', 'extra2'])
_ = bench('history.mark', lambda: G_h.history.mark('checkpoint'))
G_h.history.snapshot('before')
G_h.add_vertices(['extra3', 'extra4'])
G_h.history.snapshot('after')

_ = bench('history.list_snapshots', G_h.history.list_snapshots)
_ = bench('history.diff (before vs after)', lambda: G_h.history.diff('before', 'after'))
_ = bench('history() (full log read)', G_h.history)

In [ ]:
tmp_h = Path(tempfile.mkdtemp(prefix='annnet_hist_'))
_ = bench('history.export', lambda: G_h.history.export(tmp_h / 'hist.jsonl'))
_ = bench('history.clear', G_h.history.clear)

## 15b. Completeness sweep

Bench every remaining user-facing method (the long tail of small calls).
Most are sub-ms helpers; the goal is coverage, not optimisation.


In [ ]:
# AnnNet class-level helpers not yet measured
_ = bench('AnnNet.X (incidence sparse view)', lambda: G.X)
_ = bench('AnnNet.global_count("vertices")', lambda: G.global_count('vertices'))
_ = bench('AnnNet.number_of_vertices', lambda: G.number_of_vertices())
_ = bench('AnnNet.number_of_edges', lambda: G.number_of_edges())

In [ ]:
# G.attrs completeness
G.attrs.set_slice_attrs('s1', cohort='wt')
_ = bench('attrs.get_slice_attr', lambda: G.attrs.get_slice_attr('s1', 'cohort'))
_ = bench('attrs.set_slice_attrs', lambda: G.attrs.set_slice_attrs('s1', kind='demo'))
_ = bench(
    'attrs.get_edge_slice_attr (single)',
    lambda: G.attrs.get_edge_slice_attr('s1', sample_eids[0], 'weight'),
)

In [ ]:
# G.slices completeness
_ = bench(
    'slices.add_edge_to_slice',
    lambda: G_sl.slices.add_edge_to_slice('sl0', edge_rows[0]['edge_id']),
)
_ = bench('slices.add_vertex_to_slice', lambda: G_sl.slices.add_vertex_to_slice('sl0', v_ids[0]))
_ = bench('slices.get_slices_dict', G_sl.slices.get_slices_dict)
_ = bench(
    'slices.hyperedge_presence (by members)',
    lambda: G_sl.slices.hyperedge_presence(members=hyper_rows[0]['members']),
)
_ = bench(
    'slices.create_slice_from_operation',
    lambda: G_sl.slices.create_slice_from_operation(
        'cs_demo',
        {'vertices': set(v_ids[:50]), 'edges': set(sample_eids[:50])},
    ),
)

In [ ]:
# G.views completeness
_ = bench('views.layers_view()', G.views.layers_view)

# G.ops completeness
_ = bench('ops.get_vertex_incidence_matrix_as_lists', G.ops.get_vertex_incidence_matrix_as_lists)

In [ ]:
# G.layers completeness
_ = bench('layers.list_aspects', Gm.layers.list_aspects)
_ = bench('layers.list_layers', Gm.layers.list_layers)
_ = bench('layers.nl_to_row', lambda: Gm.layers.nl_to_row('m00000', ('L0',)))
_ = bench('layers.row_to_nl', lambda: Gm.layers.row_to_nl(0))
_ = bench('layers.layer_vertex_set', lambda: Gm.layers.layer_vertex_set(('L0',)))
_ = bench('layers.layer_edge_set', lambda: Gm.layers.layer_edge_set(('L0',)))
_ = bench('layers.ensure_vertex_layer_index', Gm.layers.ensure_vertex_layer_index)
_ = bench('layers.add_elementary_layer', lambda: Gm.layers.add_elementary_layer('x', 'L_new'))

## 16. Results table & plots


In [ ]:
import polars as pl

rows = [
    {
        'op': k,
        'wall_ms': v['wall_s'] * 1000,
        'delta_rss_mb': v['delta_rss_mb'],
        'ok': v.get('ok', True),
    }
    for k, v in _results.items()
]
df = pl.DataFrame(rows).sort('wall_ms', descending=True)
print(df)
print(f'\n{len(rows)} ops measured ({sum(1 for r in rows if not r["ok"])} skipped)')

In [ ]:
try:
    import matplotlib.pyplot as plt

    top = sorted([r for r in rows if r['ok']], key=lambda r: r['wall_ms'], reverse=True)[:30]
    plt.figure(figsize=(9, max(4, 0.32 * len(top))))
    plt.barh([r['op'] for r in top], [r['wall_ms'] for r in top])
    plt.gca().invert_yaxis()
    plt.xlabel('wall time (ms)')
    plt.title(f'annnet bench (top 30) — N={N:,}, M={M:,}')
    plt.tight_layout()
    plt.show()
except ImportError:
    print('matplotlib not available')

In [ ]:
try:
    import matplotlib.pyplot as plt

    top = sorted([r for r in rows if r['ok']], key=lambda r: abs(r['delta_rss_mb']), reverse=True)[
        :20
    ]
    plt.figure(figsize=(8, max(3, 0.35 * len(top))))
    plt.barh([r['op'] for r in top], [r['delta_rss_mb'] for r in top], color='C1')
    plt.gca().invert_yaxis()
    plt.xlabel('Δ RSS (MB)')
    plt.title(f'annnet bench memory (top 20) — N={N:,}, M={M:,}')
    plt.tight_layout()
    plt.show()
except ImportError:
    pass

## 17. Scaling probe


In [ ]:
scaling_results = []
for nn, mm in SCALING_SIZES:
    vs = [f'sv{i:07d}' for i in range(nn)]
    si = rng.integers(0, nn, size=mm)
    ti = rng.integers(0, nn, size=mm)
    er = [
        {'source': vs[s], 'target': vs[t], 'edge_id': f'se{j:08d}'}
        for j, (s, t) in enumerate(zip(si, ti, strict=False))
    ]

    t0 = time.perf_counter()
    Gp = AnnNet(directed=True)
    Gp.add_vertices(vs)
    Gp.add_edges(er)
    t_build = time.perf_counter() - t0

    t0 = time.perf_counter()
    Gp.views.edges()
    t_view = time.perf_counter() - t0

    scaling_results.append(
        {'N': nn, 'M': mm, 'construct_ms': t_build * 1000, 'views_edges_ms': t_view * 1000}
    )

pl.DataFrame(scaling_results)

---

Done. ~140 ops covering the user-facing API surface.
